# CivicGuard AI — MobileNetV2 Training (Kaggle Edition, Fixed)

This is a rebuilt version of your notebook, adapted to run on **Kaggle Notebooks** (not Colab) and fixing the issues you flagged:

| Problem you saw | What was actually happening | Fix in this notebook |
|---|---|---|
| "Epoch shows 423 but I have 11,000+ images" | `423/423` is the **batch progress bar for one epoch**, not the epoch count. Keras shows `steps/steps` while an epoch runs, and `steps = images_in_split ÷ batch_size`. With ~13,500 train images and `batch_size=32` you'd see ~`422/422` — that number was correct all along. | Section 8 prints `steps_per_epoch` explicitly and explains it, so it's never confusing again. |
| "Some datasets have PNG/JPG mixed" | Mixing formats isn't itself a problem for TensorFlow, but each dataset ships images with different color modes (RGBA, palette/P, CMYK, grayscale) and a handful of corrupt/truncated files — those *do* break `image_dataset_from_directory` or silently get mis-decoded. | Section 4 opens every image with PIL, verifies it, converts everything to clean RGB, and re-saves as `.jpg`. Corrupt files are dropped and counted for you. |
| "Datasets not fully used" | The old notebook only pulled from 4 Kaggle sources, and Colab-only code (`google.colab`) doesn't even run on Kaggle. | Section 3 pulls from **6 verified Kaggle datasets** (2 extra: an additional pothole set for `road_damage`, an additional flood set for `water_logging`) and uses every image found in each (`rglob`), not a subset. |
| "One dataset gave 0 images" | A keyword-exclude check was matching against each file's *full absolute path* — which includes the Kaggle cache folder name. The flood-mask dataset's own folder name contains the word "mask", so the check excluded the entire dataset, not just the mask images. A hardcoded `/pothole` subfolder path also didn't exist under that exact name. | Section 3b lists each dataset's real folder layout before anything is collected. Section 4's exclusion logic now only checks the path *relative to* each dataset's root, and folder lookups search case-insensitively instead of assuming an exact name. A running total + 10,000-image check is printed at the end of Section 4 so a shortfall is obvious immediately, not discovered three sections later. |
| "Different datasets for different classes" | Each source has its own folder layout (YOLO-style `images/`+`labels/`, plain folders, masked pairs, etc.) | `copy_images()` recurses through *any* layout and only cares about image files, with keyword excludes for masks. |
| Want 85–95% accuracy | A single frozen-base pass tends to plateau below that on a 3–5 class hazard set. | Two-phase training: **Phase 1** trains only the classification head, **Phase 2** unfreezes the top of MobileNetV2 and fine-tunes at a low learning rate. Class weighting handles the imbalance between classes. |
| Need documentation output | — | Section 6 (class distribution table + chart), Section 7 (sample grid), Section 14 (confusion matrix, classification report table, per-class accuracy), Section 15 (training curves) all save PNG/CSV files into `/kaggle/working/artifacts/` you can drop straight into your report. |

**Target classes (5):** `blocked_drain`, `sewage_overflow`, `road_damage`, `fallen_tree`, `water_logging`

**Honest status on `blocked_drain` / `sewage_overflow`:** I checked Kaggle and Roboflow Universe for a ready-made, download-without-a-key classification dataset for these two and couldn't find one that's clean and verified. I did **not** wire in a guessed dataset, because training on the wrong images would quietly wreck your accuracy and your report's numbers. Section 17 explains exactly what I found, the closest leads, and how to fold real data in later — the pipeline auto-detects any class folder that has images, so nothing else needs to change when you add them.

**How to run on Kaggle:** New Notebook → Settings → **Accelerator: GPU T4 x2 (or P100)** → Internet: **On** (required for `kagglehub`) → Run All.

## 1. Install dependencies

In [ ]:
# kagglehub isn't preinstalled, everything else (scikit-learn, seaborn, pillow, pandas) already is on Kaggle -
# installing only what's missing avoids version-conflict warnings with other preinstalled packages.
!pip install -q kagglehub

## 2. Imports & global config

In [ ]:
import os
import json
import shutil
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True  # don't crash on a few truncated source files, we validate separately anyway

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ["blocked_drain", "sewage_overflow", "road_damage", "fallen_tree", "water_logging"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

RAW_ROOT   = Path("/kaggle/working/data/raw")        # copied straight out of each source dataset
CLEAN_ROOT = Path("/kaggle/working/data/clean")       # validated + normalized to RGB jpg
SPLIT_ROOT = Path("/kaggle/working/data/split")       # train / val / test folders
OUTPUT_DIR = Path("/kaggle/working/artifacts")
MODEL_PATH = OUTPUT_DIR / "mobilenetv2_civicguard.keras"

for p in [RAW_ROOT, CLEAN_ROOT, SPLIT_ROOT, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

## 3. Download every source dataset

Six verified Kaggle sources feeding three of the five classes. All of this downloads automatically through `kagglehub` — nothing to add by hand. (Make sure **Internet** is switched on in the notebook's Settings panel, or these calls fail.)

In [ ]:
import kagglehub

sources = {}
sources["road_damage_1"]   = kagglehub.dataset_download("alvarobasily/road-damage")
sources["road_damage_2"]   = kagglehub.dataset_download("andrewmvd/pothole-detection")
sources["road_damage_3"]   = kagglehub.dataset_download("virenbr11/pothole-and-plain-rode-images")
sources["water_logging_1"] = kagglehub.dataset_download("saiharshitjami/flood-images-mask-segmentation")
sources["water_logging_2"] = kagglehub.dataset_download("saurabhshahane/roadway-flooding-image-dataset")
sources["fallen_tree_1"]   = kagglehub.dataset_download("akinduhiman/urban-issues-dataset")

for name, path in sources.items():
    print(f"{name:18s} -> {path}")

EXPECTED_SOURCE_KEYS = ["road_damage_1", "road_damage_2", "road_damage_3",
                         "water_logging_1", "water_logging_2", "fallen_tree_1"]
missing = [k for k in EXPECTED_SOURCE_KEYS if k not in sources]
assert not missing, (
    f"sources dict is missing {missing}. This cell (Section 3) must fully finish running before Section 4 "
    "runs — if you only re-ran later cells after an edit, go back and re-run from this cell onward."
)

## 3b. Inspect what's actually inside each download

Before writing any collection logic, look at the real folder layout Kaggle gave us. This is what catches mismatches (e.g. a subfolder that doesn't exist under the name we guessed) *before* they silently turn into "0 images collected" instead of after.

In [ ]:
def inspect_dataset(path, max_depth=2, max_entries=15):
    root = Path(path)
    print(f"\n{root}")
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        if p.is_dir():
            n_files = sum(1 for _ in p.rglob("*") if _.is_file())
            print(f"  {'  '*(depth-1)}[DIR]  {rel}/   ({n_files} files total inside)")

for name, path in sources.items():
    inspect_dataset(path)

## 4. Organize, validate, and normalize every image

Two jobs happen here, for **every** source, not a sample of it:

1. **Collect** — recursively walk each dataset's folder (whatever its layout is) and copy every image file into `data/raw/<class_name>/`, skipping mask/label images by keyword. Exclusion keywords are matched **only against the path *inside* the dataset**, never against the outer Kaggle cache path — matching the full absolute path was the bug that made the flood-mask dataset return 0 images: its own folder name contains the word "mask", so the *entire dataset* was being excluded, not just the mask files.
2. **Validate & normalize** — open each copied file with PIL, drop anything that's corrupt/zero-byte/unreadable, convert every image to RGB, and re-save as a clean `.jpg` in `data/clean/<class_name>/`. This is what actually fixes the "mixed PNG/JPG" issue: after this step every file is the same color mode and the same container format, so `image_dataset_from_directory` decodes all of them identically.

In [ ]:
def collect_images(src_dir, dest_class, prefix, exclude_keywords=None):
    """Recursively copy every image file from src_dir into data/raw/dest_class,
    skipping any file whose path *relative to src_dir* contains exclude_keywords
    (e.g. 'mask', 'label'). Matching only the relative path (not the absolute path)
    matters: the absolute path includes the dataset's own folder name, which can
    itself contain a keyword like 'mask' and wrongly exclude everything."""
    exclude_keywords = exclude_keywords or []
    src_dir = Path(src_dir)
    dest = RAW_ROOT / dest_class
    dest.mkdir(parents=True, exist_ok=True)
    count = 0
    for f in src_dir.rglob("*"):
        if not f.is_file() or f.suffix.lower() not in IMG_EXTS:
            continue
        rel_lower = str(f.relative_to(src_dir)).lower()
        if any(k in rel_lower for k in exclude_keywords):
            continue
        try:
            shutil.copy(f, dest / f"{prefix}_{count}{f.suffix.lower()}")
            count += 1
        except Exception as e:
            print(f"  skip {f}: {e}")
    flag = "  <-- 0 collected, check the Section 3b listing above" if count == 0 else ""
    print(f"Collected {count:5d} raw images -> '{dest_class}' (from {src_dir}){flag}")
    return count

def find_subdir(root, name_contains, exclude_contains=None):
    """Case-insensitive search for the first subdirectory whose name contains
    name_contains (and not exclude_contains). Falls back to root itself if
    nothing matches, so collection always scans *something* instead of a
    silently-wrong hardcoded path."""
    root = Path(root)
    exclude_contains = exclude_contains or []
    for p in sorted(root.rglob("*")):
        if p.is_dir():
            n = p.name.lower()
            if name_contains in n and not any(x in n for x in exclude_contains):
                return p
    return root

for c in CLASS_NAMES:
    (RAW_ROOT / c).mkdir(parents=True, exist_ok=True)

# road_damage <- 3 sources
collect_images(sources["road_damage_1"], "road_damage", "roaddmg")
collect_images(sources["road_damage_2"], "road_damage", "pothole")
# scan the whole dataset (layout varies) and just exclude anything under a "plain" folder/file
collect_images(sources["road_damage_3"], "road_damage", "potholeplain", exclude_keywords=["plain"])

# water_logging <- 2 sources (skip segmentation masks, keep photos only)
# exclude only files whose relative path has 'mask' in it - NOT the dataset's own folder name
collect_images(sources["water_logging_1"], "water_logging", "floodmask", exclude_keywords=["mask"])
collect_images(sources["water_logging_2"], "water_logging", "flood2")

# fallen_tree <- 1 source (multi-class YOLO bundle; only pull the FallenTrees folder)
fallen_tree_dir = find_subdir(sources["fallen_tree_1"], "fallentree")
collect_images(fallen_tree_dir, "fallen_tree", "urbanissues")

print()
print("Raw counts before cleaning:")
for c in CLASS_NAMES:
    print(f"  {c:18s} {len(list((RAW_ROOT / c).glob('*')))}")

In [ ]:
def validate_and_clean(src_class_dir, dest_class_dir, target_ext=".jpg", min_size=32):
    """Open every image, drop corrupt/too-small ones, convert to RGB, save clean copies."""
    dest_class_dir.mkdir(parents=True, exist_ok=True)
    kept, dropped = 0, 0
    for f in sorted(src_class_dir.glob("*")):
        try:
            with Image.open(f) as im:
                im.verify()  # cheap structural check first
            with Image.open(f) as im:
                if im.width < min_size or im.height < min_size:
                    dropped += 1
                    continue
                im = im.convert("RGB")
                out_path = dest_class_dir / f"{f.stem}{target_ext}"
                im.save(out_path, "JPEG", quality=92)
                kept += 1
        except Exception:
            dropped += 1
    return kept, dropped

print("Validating + normalizing images (this can take a few minutes for 10k+ files)...\n")
clean_counts = {}
for c in CLASS_NAMES:
    src = RAW_ROOT / c
    dest = CLEAN_ROOT / c
    if not src.exists() or not any(src.iterdir()):
        clean_counts[c] = 0
        print(f"  {c:18s} -> no raw images, skipping")
        continue
    kept, dropped = validate_and_clean(src, dest)
    clean_counts[c] = kept
    print(f"  {c:18s} kept {kept:5d}  |  dropped (corrupt/too small) {dropped}")

print()
print("Clean, ready-to-train counts:")
for c, n in clean_counts.items():
    print(f"  {c:18s} {n}")

total_images = sum(clean_counts.values())
print(f"\nTOTAL across all classes: {total_images}")
if total_images < 10000:
    print(f"⚠ That's below the 10,000-image target. If a class above shows 0 or looks too low, "
          f"scroll up to the Section 3b listing to see what that dataset's real folder layout is, "
          f"then adjust the matching keyword/folder search in Section 4 accordingly.")
else:
    print("✅ Past the 10,000-image target.")

## 5. Auto-select classes that actually have data

Exactly like before: any class with zero images after cleaning is skipped automatically, and picked back up the moment you add data for it (Section 17).

In [ ]:
MIN_IMAGES_PER_CLASS = 20  # below this, a class isn't reliable to train/evaluate on

ACTIVE_CLASSES = [c for c in CLASS_NAMES if clean_counts.get(c, 0) >= MIN_IMAGES_PER_CLASS]
SKIPPED = [c for c in CLASS_NAMES if c not in ACTIVE_CLASSES]

print("Training on:", ACTIVE_CLASSES)
if SKIPPED:
    print("Skipped (not enough images yet):", SKIPPED)

assert len(ACTIVE_CLASSES) >= 2, "Need at least 2 classes with data to train a classifier."


## 6. Documentation output #1 — class distribution table & chart

Straight from the cleaned dataset, saved to `artifacts/` for your report.

In [ ]:
dist_df = pd.DataFrame({
    "class": ACTIVE_CLASSES,
    "image_count": [clean_counts[c] for c in ACTIVE_CLASSES],
})
dist_df["share_%"] = (dist_df["image_count"] / dist_df["image_count"].sum() * 100).round(1)
dist_df = dist_df.sort_values("image_count", ascending=False).reset_index(drop=True)
dist_df.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
dist_df

In [ ]:
plt.figure(figsize=(8, 4.5))
bars = plt.bar(dist_df["class"], dist_df["image_count"], color="#3b7dd8")
plt.title("CivicGuard AI — Images per Class (after cleaning)")
plt.ylabel("Image count")
plt.xticks(rotation=20)
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 5, int(b.get_height()),
              ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150)
plt.show()

imbalance_ratio = dist_df["image_count"].max() / dist_df["image_count"].min()
print(f"Largest / smallest class ratio: {imbalance_ratio:.2f}x  "
      f"({'will use class weighting to compensate' if imbalance_ratio > 1.5 else 'roughly balanced already'})")

## 7. Documentation output #2 — sample images per class

In [ ]:
fig, axes = plt.subplots(len(ACTIVE_CLASSES), 4, figsize=(12, 3 * len(ACTIVE_CLASSES)))
if len(ACTIVE_CLASSES) == 1:
    axes = np.expand_dims(axes, 0)

for row, c in enumerate(ACTIVE_CLASSES):
    files = sorted((CLEAN_ROOT / c).glob("*"))[:4]
    for col in range(4):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(files):
            img = Image.open(files[col])
            ax.imshow(img)
        if col == 0:
            ax.set_ylabel(c, fontsize=10)
        if row == 0:
            ax.set_title(f"sample {col+1}", fontsize=9)

plt.suptitle("CivicGuard AI — Sample Images per Class", y=1.0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_grid.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Train / validation / test split (physical folders)

A proper **70 / 15 / 15** stratified split instead of a single validation split — the test set is never touched until final evaluation in Section 14, which gives you an honest accuracy number instead of one inflated by repeated peeking at validation data during training.

In [ ]:
if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)
for subset in ["train", "val", "test"]:
    for c in ACTIVE_CLASSES:
        (SPLIT_ROOT / subset / c).mkdir(parents=True, exist_ok=True)

split_summary = []
for c in ACTIVE_CLASSES:
    files = sorted((CLEAN_ROOT / c).glob("*"))
    train_files, temp_files = train_test_split(files, test_size=0.30, random_state=SEED)
    val_files, test_files = train_test_split(temp_files, test_size=0.50, random_state=SEED)

    for subset, subset_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
        for f in subset_files:
            shutil.copy(f, SPLIT_ROOT / subset / c / f.name)

    split_summary.append({"class": c, "train": len(train_files), "val": len(val_files), "test": len(test_files)})

split_df = pd.DataFrame(split_summary)
split_df.to_csv(OUTPUT_DIR / "train_val_test_split.csv", index=False)
split_df

## 9. Load datasets — and a clear look at steps vs. epochs

This is the cell that clears up the "epoch shows 423" confusion. The progress bar Keras prints during `model.fit` looks like:

```
183/422 [==============>...............] - ETA: 12s - loss: 0.41 - accuracy: 0.85
```

`422` here is **`steps_per_epoch`** — the number of *batches* in one epoch, computed as `training_images ÷ batch_size`. It is printed on every single epoch, so seeing "423" doesn't mean you only trained for 423 total images or that training stopped at epoch 423 — it's just how many batches Keras chews through *per* epoch. The actual epoch count is whatever you set in `model.fit(epochs=...)`, printed separately as `Epoch 1/10`, `Epoch 2/10`, etc.

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    SPLIT_ROOT / "train", seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
val_ds = keras.utils.image_dataset_from_directory(
    SPLIT_ROOT / "val", seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)
test_ds = keras.utils.image_dataset_from_directory(
    SPLIT_ROOT / "test", seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)

class_names = train_ds.class_names
print("class_names:", class_names)

n_train = sum(1 for _ in (SPLIT_ROOT / "train").rglob("*") if _.is_file())
n_val   = sum(1 for _ in (SPLIT_ROOT / "val").rglob("*") if _.is_file())
n_test  = sum(1 for _ in (SPLIT_ROOT / "test").rglob("*") if _.is_file())
steps_per_epoch = -(-n_train // BATCH_SIZE)  # ceil division, matches what Keras will print
val_steps = -(-n_val // BATCH_SIZE)

print(f"\nTrain images: {n_train}  ->  steps_per_epoch = {steps_per_epoch}  "
      f"(this is the number you'll see as '{steps_per_epoch}/{steps_per_epoch}' in the progress bar)")
print(f"Val images:   {n_val}  ->  validation_steps = {val_steps}")
print(f"Test images:  {n_test}  (held out, only used in Section 14)")

AUTOTUNE = tf.data.AUTOTUNE
train_ds_cached = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds_cached = val_ds.cache().prefetch(AUTOTUNE)
test_ds_cached = test_ds.cache().prefetch(AUTOTUNE)

## 10. Class weights (compensate for imbalance)

Instead of silently letting the biggest class dominate the loss, we compute a weight per class from the actual training counts and pass it into `model.fit`.

In [ ]:
train_labels = []
for c_idx, c in enumerate(class_names):
    n = len(list((SPLIT_ROOT / "train" / c).glob("*")))
    train_labels += [c_idx] * n

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=np.array(train_labels),
)
class_weight = {i: float(w) for i, w in enumerate(class_weight_values)}

cw_df = pd.DataFrame({"class": class_names, "train_images": [train_labels.count(i) for i in range(len(class_names))],
                       "class_weight": [round(class_weight[i], 3) for i in range(len(class_names))]})
cw_df.to_csv(OUTPUT_DIR / "class_weights.csv", index=False)
cw_df

## 11. Build the model — MobileNetV2 transfer learning

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="augmentation")

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # frozen for Phase 1

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

## 12. Phase 1 — train the classification head (base frozen)

In [ ]:
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_phase1 = model.fit(
    train_ds_cached,
    validation_data=val_ds_cached,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks_phase1,
)

## 13. Phase 2 — fine-tune the top of MobileNetV2

Unfreeze the last ~40 layers of the backbone and continue training at a much lower learning rate. This is usually what pushes a transfer-learning model from the "mid-70s%" range up into the 85–95% range on a small hazard dataset like this — the frozen-base features are generic ImageNet features, fine-tuning lets the top layers specialize on drains/roads/trees/floods.

In [ ]:
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 40  # unfreeze only the last 40 layers
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint(str(MODEL_PATH), monitor="val_accuracy", save_best_only=True),
]

FINE_TUNE_EPOCHS = 15
total_prior_epochs = len(history_phase1.history["loss"])

history_phase2 = model.fit(
    train_ds_cached,
    validation_data=val_ds_cached,
    epochs=total_prior_epochs + FINE_TUNE_EPOCHS,
    initial_epoch=total_prior_epochs,
    class_weight=class_weight,
    callbacks=callbacks_phase2,
)

## 14. Evaluate on the held-out test set + documentation output #3

This is the number that matters — the test set was never seen during training or used for early stopping.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds_cached)
print(f"\nTest loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

y_true, y_pred = [], []
for batch_images, batch_labels in test_ds:
    preds = model.predict(batch_images, verbose=0)
    y_true.extend(batch_labels.numpy().tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().round(3)
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")
report_df

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — Test Accuracy {test_accuracy*100:.1f}%")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

per_class_acc = (cm.diagonal() / cm.sum(axis=1)).round(3)
per_class_df = pd.DataFrame({"class": class_names, "test_images": cm.sum(axis=1), "accuracy": per_class_acc})
per_class_df.to_csv(OUTPUT_DIR / "per_class_accuracy.csv", index=False)
per_class_df

## 15. Documentation output #4 — full training curves (Phase 1 + Phase 2)

In [ ]:
def stitch(key):
    return history_phase1.history[key] + history_phase2.history[key]

acc = stitch("accuracy"); val_acc = stitch("val_accuracy")
loss = stitch("loss"); val_loss = stitch("val_loss")
phase_boundary = len(history_phase1.history["loss"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(acc, label="train")
axes[0].plot(val_acc, label="val")
axes[0].axvline(phase_boundary - 0.5, color="gray", linestyle="--", label="fine-tuning starts")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(loss, label="train")
axes[1].plot(val_loss, label="val")
axes[1].axvline(phase_boundary - 0.5, color="gray", linestyle="--", label="fine-tuning starts")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 16. Save model, labels, and a metrics summary

In [ ]:
model.save(MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")

labels_path = OUTPUT_DIR / "class_names.json"
labels_path.write_text(json.dumps(class_names, indent=2), encoding="utf-8")

summary = {
    "classes_trained": class_names,
    "classes_skipped_no_data": SKIPPED,
    "train_images": n_train,
    "val_images": n_val,
    "test_images": n_test,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "phase1_epochs_run": len(history_phase1.history["loss"]),
    "phase2_epochs_run": len(history_phase2.history["loss"]),
    "class_weights": class_weight,
}
summary_path = OUTPUT_DIR / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("\nAll artifacts saved under:", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob("*")):
    print(" -", f.name)

> **Getting your files off Kaggle:** unlike Colab, Kaggle doesn't auto-download. Everything is saved under `/kaggle/working/artifacts/` — after you run the notebook, use the **Output** tab (or "Save Version" → "Save & Run All") and Kaggle packages that whole folder for you to download as a zip: model, labels, every CSV table, and every PNG chart above, ready to paste into your documentation.

## 17. Adding `blocked_drain` and `sewage_overflow`

**What I checked:** Kaggle dataset search and Roboflow Universe for a ready classification dataset of blocked-drain / sewage-overflow photos. Nothing came back that's both (a) an actual classification set of real photos (not object-detection boxes, not a CSV of overflow event timestamps, not a sewer-pipe CCTV/point-cloud engineering set) and (b) downloadable without a personal Roboflow API key. So none is wired into Section 3 — a guessed dataset here would quietly corrupt those two classes' training data.

**Closest real leads, for you to check manually:**
- Roboflow Universe search for [`drain`](https://universe.roboflow.com/search?q=class%3Adrain) — several community object-detection sets tag `drain`, `manhole`, `blocked drain` as *boxes*, not clean classification folders. You'd need to crop the boxes into images per class yourself, and it needs a free Roboflow account + API key.
- `myprojectdata/drainage-and-waterbody-datasetchennai` on Kaggle — unverified content, worth a manual look.
- For `sewage_overflow`, most of what's on Kaggle is sewer-pipe *inspection/defect* data (point clouds, CCTV defect classification for cracks/roots — not street-level overflow photos), which isn't the same hazard your class is meant to capture.

**Fastest realistic path:** shoot or collect 100–150 real photos per class yourself (civic hazard photos travel well — angles from ground level, daytime, varied backgrounds), zip them, and either:

```python
# Option A: upload the zip via Kaggle's "+ Add Data" button, then:
import zipfile
with zipfile.ZipFile("/kaggle/input/your-dataset-name/blocked_drain.zip", "r") as z:
    z.extractall(RAW_ROOT / "blocked_drain")
```

```python
# Option B: if you host it as your own Kaggle dataset, one line, same as the sources in Section 3:
sources["blocked_drain_1"] = kagglehub.dataset_download("your-username/blocked-drain-photos")
collect_images(sources["blocked_drain_1"], "blocked_drain", "custom1")
```

Either way, just re-run from **Section 4** onward — validation, cleaning, class weighting, the split, and both training phases all auto-detect the newly non-empty folder and retrain on all 5 classes with no other changes needed.